# Capstone Project: Predicting High-Value Customers
## Step 2 — Data Wrangling

**Business Problem:** A retail brand (BigMart) spends marketing budget equally across all customers.
This notebook performs data wrangling on the *Customer Shopping Trends Dataset* (~3,900 customers)
to prepare it for building a customer-value segmentation model (High / Medium / Low).

This notebook follows the **Data Science Method (DSM)** wrangling sub-steps:
1. **Data Collection** — load the raw dataset and record its source/shape
2. **Data Organization** — set up a clean project folder structure and save raw/working copies
3. **Data Definition** — define each column, its data type, and units; produce a data dictionary
4. **Data Cleaning** — check & handle missing values, duplicates, inconsistent categories, and outliers
5. **Feature Engineering (light)** — create the proxy "high-value" label needed because the dataset
   has no Customer Lifetime Value (CLV) field 

## 1. Data Collection

Load the raw CSV and take a first look at its shape, columns, and a sample of rows.

In [1]:
import pandas as pd
import numpy as np
import os

# Path to the raw dataset (downloaded from Kaggle:
# https://www.kaggle.com/datasets/iamsouravbanerjee/customer-shopping-trends-dataset)
RAW_PATH = "shopping_trends.csv"

df_raw = pd.read_csv(RAW_PATH)

print("Shape (rows, columns):", df_raw.shape)
df_raw.head()


Shape (rows, columns): (3900, 19)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Payment Method,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Preferred Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Credit Card,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Bank Transfer,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Cash,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,PayPal,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Cash,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
# Quick look at column names and data types as loaded by pandas
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer ID               3900 non-null   int64  
 1   Age                       3900 non-null   int64  
 2   Gender                    3900 non-null   object 
 3   Item Purchased            3900 non-null   object 
 4   Category                  3900 non-null   object 
 5   Purchase Amount (USD)     3900 non-null   int64  
 6   Location                  3900 non-null   object 
 7   Size                      3900 non-null   object 
 8   Color                     3900 non-null   object 
 9   Season                    3900 non-null   object 
 10  Review Rating             3900 non-null   float64
 11  Subscription Status       3900 non-null   object 
 12  Payment Method            3900 non-null   object 
 13  Shipping Type             3900 non-null   object 
 14  Discount

## 2. Data Organization

Create a simple, reproducible folder structure so raw data, cleaned data, and notebooks
stay separated. We never overwrite the raw file -- all cleaning happens on a copy.


In [3]:
# Create folders for organizing raw vs. processed data
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Save an untouched copy of the raw data into data/raw (if not already there)
raw_copy_path = "data/raw/shopping_trends_raw.csv"
if not os.path.exists(raw_copy_path):
    df_raw.to_csv(raw_copy_path, index=False)

# All cleaning is performed on a working copy, never on df_raw directly
df = df_raw.copy()
print("Working copy created with shape:", df.shape)


Working copy created with shape: (3900, 19)


## 3. Data Definition

Before cleaning, we define what each column represents, its data type, and any constraints.
This "data dictionary" step makes later cleaning decisions easier to justify and documents the
dataset for anyone else reading the notebook.


In [4]:
data_dictionary = pd.DataFrame([
    ("Customer ID",              "int",   "Unique identifier for each customer"),
    ("Age",                       "int",   "Customer age in years"),
    ("Gender",                    "category", "Male / Female"),
    ("Item Purchased",            "category", "Specific item bought (e.g. Blouse, Sweater)"),
    ("Category",                  "category", "Product category (Clothing, Footwear, Outerwear, Accessories)"),
    ("Purchase Amount (USD)",     "int",   "Amount spent on the purchase, in USD"),
    ("Location",                  "category", "US state of the customer"),
    ("Size",                      "category", "Item size (S, M, L, XL)"),
    ("Color",                     "category", "Item color"),
    ("Season",                    "category", "Season the purchase was made in"),
    ("Review Rating",             "float", "Customer's review rating for the item (1-5 scale)"),
    ("Subscription Status",       "category", "Whether the customer is subscribed (Yes/No)"),
    ("Payment Method",            "category", "Payment method used for this transaction"),
    ("Shipping Type",             "category", "Shipping method chosen"),
    ("Discount Applied",          "category", "Whether a discount was applied (Yes/No)"),
    ("Promo Code Used",           "category", "Whether a promo code was used (Yes/No)"),
    ("Previous Purchases",        "int",   "Count of previous purchases by this customer"),
    ("Preferred Payment Method",  "category", "Customer's generally preferred payment method"),
    ("Frequency of Purchases",    "category", "How often the customer typically shops"),
], columns=["Column", "Data Type", "Description"])

data_dictionary


,Column,Data Type,Description
0,Customer ID,int,Unique identifier for each customer
1,Age,int,Customer age in years
2,Gender,category,Male / Female
3,Item Purchased,category,"Specific item bought (e.g. Blouse, Sweater)"
4,Category,category,"Product category (Clothing, Footwear, Outerwea..."
5,Purchase Amount (USD),int,"Amount spent on the purchase, in USD"
6,Location,category,US state of the customer
7,Size,category,"Item size (S, M, L, XL)"
8,Color,category,Item color
9,Season,category,Season the purchase was made in


## 4. Data Cleaning

We work through the standard checks: missing values, duplicate records, inconsistent
categorical labels, valid value ranges, and outliers. Each check is documented with the
finding and the decision made.


### 4.1 Missing values

In [5]:
# Count missing (NaN) values per column
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary[missing_summary["missing_count"] > 0]


,missing_count,missing_pct


**Finding:** This dataset, as sourced, contains **no missing values** in any column
(every cell is populated). The project's problem statement anticipated possible NaNs, so
this check was performed explicitly and the result -- zero missing values -- is documented
here as a data-supported decision: **no imputation or row-dropping is required for missingness.**

We still write a small reusable imputation block below (commented out / guarded) so that if
a refreshed pull of the dataset *does* introduce missing values later, the notebook handles
it without manual edits.


In [7]:
# Defensive imputation step (only runs if missing values are actually present)
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object"]).columns

if df.isnull().values.any():
    # Numeric columns -> fill with column median (robust to outliers)
    for c in numeric_cols:
        if df[c].isnull().any():
            df[c] = df[c].fillna(df[c].median())
    # Categorical columns -> fill with column mode (most frequent value)
    for c in categorical_cols:
        if df[c].isnull().any():
            df[c] = df[c].fillna(df[c].mode()[0])
    print("Missing values were found and imputed.")
else:
    print("No missing values found -- no imputation needed.")


No missing values found -- no imputation needed.


### 4.2 Duplicate records

In [8]:
# Check for fully duplicated rows
n_full_dupes = df.duplicated().sum()
print("Fully duplicated rows:", n_full_dupes)

# Check for duplicated Customer IDs (each customer should appear once)
n_dupe_ids = df["Customer ID"].duplicated().sum()
print("Duplicated Customer IDs:", n_dupe_ids)

if n_full_dupes > 0:
    df = df.drop_duplicates()
    print("Dropped", n_full_dupes, "duplicate rows. New shape:", df.shape)
else:
    print("No duplicate rows found -- no rows dropped.")


Fully duplicated rows: 0
Duplicated Customer IDs: 0
No duplicate rows found -- no rows dropped.


**Finding:** No fully duplicated rows and no duplicated `Customer ID` values were found.
Each row represents a unique customer record, which matches the problem statement's framing
of "~3,900 customers". **Decision: keep all rows as-is.**


### 4.3 Consistency of categorical values

In [9]:
# For each categorical column, inspect the unique values for inconsistent labels
# (e.g. mismatched capitalization, trailing whitespace, typo variants like "Yes"/"yes "/"YES")
categorical_cols_to_check = [
    "Gender", "Category", "Season", "Subscription Status", "Payment Method",
    "Shipping Type", "Discount Applied", "Promo Code Used",
    "Frequency of Purchases", "Size"
]

for col in categorical_cols_to_check:
    print(f"{col}: {sorted(df[col].unique())}")


Gender: ['Female', 'Male']
Category: ['Accessories', 'Clothing', 'Footwear', 'Outerwear']
Season: ['Fall', 'Spring', 'Summer', 'Winter']
Subscription Status: ['No', 'Yes']
Payment Method: ['Bank Transfer', 'Cash', 'Credit Card', 'Debit Card', 'PayPal', 'Venmo']
Shipping Type: ['2-Day Shipping', 'Express', 'Free Shipping', 'Next Day Air', 'Standard', 'Store Pickup']
Discount Applied: ['No', 'Yes']
Promo Code Used: ['No', 'Yes']
Frequency of Purchases: ['Annually', 'Bi-Weekly', 'Every 3 Months', 'Fortnightly', 'Monthly', 'Quarterly', 'Weekly']
Size: ['L', 'M', 'S', 'XL']


**Finding:** All categorical columns contain a small, clean set of consistently-formatted
values (consistent capitalization, no leading/trailing whitespace, no duplicate-meaning
categories such as "Yes" vs "yes"). **Decision: no relabeling or standardization needed.**

As a safeguard, we still strip whitespace and standardize casing across all text columns below,
which is a no-op here but protects against issues if the dataset is refreshed.


In [10]:
# Defensive standardization: strip whitespace and fix casing on all object columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Title-case the Yes/No style flag columns for consistency
flag_cols = ["Subscription Status", "Discount Applied", "Promo Code Used"]
for col in flag_cols:
    df[col] = df[col].str.title()

print("Standardization applied. Example unique values after cleanup:")
print("Subscription Status:", df["Subscription Status"].unique())


Standardization applied. Example unique values after cleanup:
Subscription Status: ['Yes' 'No']


### 4.4 Valid ranges and outliers (numeric columns)

In [11]:
numeric_cols_to_check = ["Age", "Purchase Amount (USD)", "Review Rating", "Previous Purchases"]
df[numeric_cols_to_check].describe()


,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3900.000000
mean,44.068462,59.764359,3.749949,25.351538
std,15.207589,23.685392,0.716223,14.447125
min,18.000000,20.000000,2.500000,1.000000
25%,31.000000,39.000000,3.100000,13.000000
50%,44.000000,60.000000,3.700000,25.000000
75%,57.000000,81.000000,4.400000,38.000000
max,70.000000,100.000000,5.000000,50.000000


In [12]:
# Check for logically invalid values:
# - Age should be a realistic adult age (e.g. 18-100)
# - Purchase Amount should be positive
# - Review Rating should be within 1-5
# - Previous Purchases should be >= 0

issues = {
    "Age out of [18, 100]":            ((df["Age"] < 18) | (df["Age"] > 100)).sum(),
    "Purchase Amount <= 0":            (df["Purchase Amount (USD)"] <= 0).sum(),
    "Review Rating out of [1, 5]":     ((df["Review Rating"] < 1) | (df["Review Rating"] > 5)).sum(),
    "Previous Purchases < 0":          (df["Previous Purchases"] < 0).sum(),
}

for k, v in issues.items():
    print(f"{k}: {v} row(s)")


Age out of [18, 100]: 0 row(s)
Purchase Amount <= 0: 0 row(s)
Review Rating out of [1, 5]: 0 row(s)
Previous Purchases < 0: 0 row(s)


**Finding:** No rows violate these logical bounds. Age, Purchase Amount, Review Rating,
and Previous Purchases all fall within expected, realistic ranges. **Decision: no rows need
to be removed or capped for invalid values.**


In [13]:
# Outlier check using the IQR method on the key spending-related numeric columns
def iqr_outlier_count(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum(), lower, upper

for col in ["Purchase Amount (USD)", "Previous Purchases", "Age"]:
    n_out, lo, hi = iqr_outlier_count(df[col])
    print(f"{col}: {n_out} outlier(s) outside [{lo:.1f}, {hi:.1f}]")


Purchase Amount (USD): 0 outlier(s) outside [-24.0, 144.0]
Previous Purchases: 0 outlier(s) outside [-24.5, 75.5]
Age: 0 outlier(s) outside [-8.0, 96.0]


**Finding:** The IQR method flags **0 outliers** for Purchase Amount, Previous Purchases,
and Age -- all values lie within 1.5x the interquartile range. This is consistent with the
dataset being a clean, capped sample (Purchase Amount ranges 20-100, Age 18-70, Previous
Purchases 1-50). **Decision: no outlier removal or transformation needed.**


## 5. Feature Engineering: Proxy "Customer Value Tier" Label

**Constraint from the problem statement:** the dataset has no Customer Lifetime Value (CLV)
label, so a proxy metric is needed to define High / Medium / Low value segments.

We construct a simple, transparent **Customer Value Score** as a weighted combination of:
- `Purchase Amount (USD)` -- how much they spend per transaction
- `Previous Purchases` -- how often/long they've been purchasing
- `Subscription Status` -- subscribers are typically more engaged/retained

These three map directly to the "Scope of solution space" defined in the problem statement
(purchase amount, purchase frequency/previous purchases, subscription status).


In [14]:
# Step 1: scale the two numeric drivers to 0-1 so they contribute comparably
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

df["purchase_amount_scaled"] = min_max_scale(df["Purchase Amount (USD)"])
df["previous_purchases_scaled"] = min_max_scale(df["Previous Purchases"])

# Step 2: convert subscription status to a numeric flag
df["is_subscribed"] = (df["Subscription Status"] == "Yes").astype(int)

# Step 3: combine into a single weighted "Customer Value Score" (0-1 range, higher = more valuable)
# Weights reflect relative importance for marketing ROI:
#   spend amount (45%), purchase history/loyalty (35%), active subscription (20%)
df["customer_value_score"] = (
    0.45 * df["purchase_amount_scaled"] +
    0.35 * df["previous_purchases_scaled"] +
    0.20 * df["is_subscribed"]
)

df[["Customer ID", "Purchase Amount (USD)", "Previous Purchases",
    "Subscription Status", "customer_value_score"]].head()


,Customer ID,Purchase Amount (USD),Previous Purchases,Subscription Status,customer_value_score
0,1,53,14,Yes,0.478482
1,2,64,2,Yes,0.454643
2,3,73,23,Yes,0.655268
3,4,90,49,Yes,0.936607
4,5,49,31,Yes,0.577411


In [15]:
# Step 4: bucket the continuous score into High / Medium / Low tiers using terciles
# (each tier holds roughly one-third of customers, giving balanced classes for modeling later)
df["customer_value_tier"] = pd.qcut(
    df["customer_value_score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df["customer_value_tier"].value_counts()


customer_value_tier
Low       1303
High      1300
Medium    1297
Name: count, dtype: int64

In [16]:
# Sanity check: confirm the tiers behave as expected (High tier should have higher
# average spend, previous purchases, and subscription rate than Low tier)
df.groupby("customer_value_tier")[
    ["Purchase Amount (USD)", "Previous Purchases", "is_subscribed"]
].mean().round(2)


/var/folders/kq/wf2fxs3j0j1_z43cz83vpkp00000gp/T/ipykernel_27511/4250449780.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("customer_value_tier")[


,Purchase Amount (USD),Previous Purchases,is_subscribed
customer_value_tier,,,
Low,40.35,16.83,0.07
Medium,61.59,24.77,0.22
High,77.40,34.48,0.52


**Result:** The High tier shows clearly higher average purchase amount, previous purchase
count, and subscription rate compared to Low -- confirming the proxy score behaves sensibly
and can serve as the target label for a customer-value classification model in the next step.


## 6. Save the Cleaned Dataset

The cleaned dataset, including the new `customer_value_score` and `customer_value_tier`
columns, is saved to `data/processed/` for use in the next (EDA / modeling) notebooks.


In [17]:
processed_path = "data/processed/shopping_trends_cleaned.csv"
df.to_csv(processed_path, index=False)

print("Saved cleaned dataset to:", processed_path)
print("Final shape:", df.shape)
df.head()


Saved cleaned dataset to: data/processed/shopping_trends_cleaned.csv
Final shape: (3900, 24)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Discount Applied,Promo Code Used,Previous Purchases,Preferred Payment Method,Frequency of Purchases,purchase_amount_scaled,previous_purchases_scaled,is_subscribed,customer_value_score,customer_value_tier
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Yes,Yes,14,Venmo,Fortnightly,0.4125,0.265306,1,0.478482,Medium
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Yes,Yes,2,Cash,Fortnightly,0.5500,0.020408,1,0.454643,Medium
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Yes,Yes,23,Credit Card,Weekly,0.6625,0.448980,1,0.655268,High
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Yes,Yes,49,PayPal,Weekly,0.8750,0.979592,1,0.936607,High
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Yes,Yes,31,PayPal,Annually,0.3625,0.612245,1,0.577411,High


## 7. Summary of Data Wrangling Decisions

| Step | Check | Finding | Decision |
|---|---|---|---|
| Missing values | `isnull()` per column | 0 missing values across all 19 columns | No imputation needed (defensive imputation code included for future refreshes) |
| Duplicates | Full-row and `Customer ID` duplicates | 0 duplicate rows, 0 duplicate IDs | Kept all 3,900 rows |
| Categorical consistency | Unique values per categorical column | All values consistently formatted | No relabeling needed; defensive whitespace/casing cleanup applied |
| Valid ranges | Age, Purchase Amount, Review Rating, Previous Purchases bounds | All values within realistic bounds | No rows removed |
| Outliers | IQR method on spend/loyalty columns | 0 outliers detected | No capping/removal needed |
| CLV proxy (constraint) | No CLV column available | N/A | Engineered `customer_value_score` (weighted: spend 45%, history 35%, subscription 20%) and `customer_value_tier` (High/Medium/Low via terciles) |

The dataset is now clean, well-defined, and includes a proxy target label (`customer_value_tier`)
ready for exploratory data analysis and predictive modeling in the next phase of the project.
